# Process APT campaigns

## Label the dataset
For each campaign, there is a dedicated folder containing:
- a _flows.csv file converted from the raw .pcap file, containing extracted flows.
- a .pkl file containing detailed information on the executed attack steps.
- a .npz file containing a datetime object that mark the start of the experiment.

In [51]:
# Make project helpers available from a kernel started in this folder or the repo root.
from pathlib import Path
import sys
PROJECT_ROOT = next(root for root in (Path.cwd(), *Path.cwd().parents)
                    if (root / "preprocessing").is_dir() and (root / "scenario_reconstructor").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from datetime import datetime
import pytz
import pickle

In [52]:
ATTACK_DATA_ROOT = PROJECT_ROOT / "data" / "cAPTure" / "attack"
campaign_names = ["dollar_char"]
campaign_ids = {name: idx for idx, name in enumerate(campaign_names)}

In [53]:
root_list = []
csv_file_list = []
dtls_list = []
start_list = []
for name in campaign_names:
    train_dir = ATTACK_DATA_ROOT / f"{name}_train"
    test_dir = ATTACK_DATA_ROOT / f"{name}_test"
    root_list.append((train_dir, test_dir))
    csv_file_list.append((train_dir / f"{train_dir.name}_flows.csv", test_dir / f"{test_dir.name}_flows.csv"))
    dtls_list.append((train_dir / f"{train_dir.name}.pkl", test_dir / f"{test_dir.name}.pkl"))
    start_list.append((train_dir / f"{train_dir.name}.npz", test_dir / f"{test_dir.name}.npz"))

### Load attack step detail files
We load the attack_step_details object, containing essential information about each attack step for its attack campaign.

In [54]:
step_details_list = []
for train_step_details, test_step_details in dtls_list:
    with open(train_step_details, "rb") as f1, open(test_step_details, "rb") as f2:
        step_details_list.append((pickle.load(f1), pickle.load(f2)))

In [55]:
for name, (train_data, test_data) in zip(campaign_names, step_details_list):
    print(f"{name} train attack step number: {len(train_data)}")
    print(f"{name} test attack step number: {len(test_data)}")

dollar_char train timestamp obj size: 533
dollar_char test timestamp obj size: 310


In [56]:
for name, (train_data, test_data) in zip(campaign_names, step_details_list):
    print(f"=== {name} train (first 2) ===", train_data[:2])
    print(f"=== {name} test (first 2) ===", test_data[:2])

=== dollar_char train (first 2) === [{'phase_number': 1, 'step_number': 1, 'attack_name': 'dollar_char_dos_scp_inst_cam_train', 'phase_name': 'RECONNAISSANCE', 'start_time': datetime.datetime(2025, 8, 19, 19, 11, 16, 388096, tzinfo=<UTC>), 'end_time': datetime.datetime(2025, 8, 19, 19, 11, 16, 576507, tzinfo=<UTC>), 'command': 'netstat', 'iteration': 0}, {'phase_number': -1, 'step_number': 2, 'attack_name': 'dollar_char_dos_scp_inst_cam_train', 'phase_name': 'PAUSE', 'start_time': datetime.datetime(2025, 8, 19, 19, 11, 16, 576560, tzinfo=<UTC>), 'end_time': datetime.datetime(2025, 8, 19, 19, 12, 23, 815582, tzinfo=<UTC>), 'command': 'pause', 'iteration': 0}]
=== dollar_char test (first 2) === [{'phase_number': 1, 'step_number': 1, 'attack_name': 'dollar_char_dos_scp_inst_cam_test', 'phase_name': 'RECONNAISSANCE', 'start_time': datetime.datetime(2025, 8, 20, 11, 38, 18, 722238, tzinfo=<UTC>), 'end_time': datetime.datetime(2025, 8, 20, 11, 38, 18, 881681, tzinfo=<UTC>), 'command': 'netst

### Load CSV network traffic files
We load the network flow dataset obtained by estracting network flows from the initial packet dataset through the convert_to_csv notebook.

In [57]:
df_list = []
for train_csv, test_csv in csv_file_list:
    df_list.append((pd.read_csv(train_csv, low_memory=False), pd.read_csv(test_csv, low_memory=False)))

/tmp/ipykernel_1564089/169112827.py:3: DtypeWarning: Columns (0: requested_server_name, 1: client_fingerprint, 2: server_fingerprint) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list.append((pd.read_csv(train_csv), pd.read_csv(test_csv)))


In [58]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"{name} train dataframe size: {train_df.shape}")
    print(f"{name} test dataframe size: {test_df.shape}")

dollar_char train dataframe size: (198928, 86)
dollar_char test dataframe size: (115219, 86)


In [59]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"{name} train columns:", list(train_df.columns))
    print(f"{name} test columns:", list(test_df.columns))

dollar_char train columns: ['id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui', 'src_port', 'dst_ip', 'dst_mac', 'dst_oui', 'dst_port', 'protocol', 'ip_version', 'vlan_id', 'tunnel_id', 'bidirectional_first_seen_ms', 'bidirectional_last_seen_ms', 'bidirectional_duration_ms', 'bidirectional_packets', 'bidirectional_bytes', 'src2dst_first_seen_ms', 'src2dst_last_seen_ms', 'src2dst_duration_ms', 'src2dst_packets', 'src2dst_bytes', 'dst2src_first_seen_ms', 'dst2src_last_seen_ms', 'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes', 'bidirectional_min_ps', 'bidirectional_mean_ps', 'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps', 'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps', 'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps', 'dst2src_max_ps', 'bidirectional_min_piat_ms', 'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 'src2dst_min_piat_ms', 'src2dst_mean_piat_ms', 'src2dst_stddev_piat_ms', 'src2dst_max

In [60]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"=== {name} train bidirectional_first_seen_ms ===")
    print(train_df["bidirectional_first_seen_ms"].head())
    print(f"=== {name} test bidirectional_first_seen_ms ===")
    print(test_df["bidirectional_first_seen_ms"].head())

=== dollar_char train bidirectional_first_seen_ms ===
0     10
1    348
2    412
3    460
4    476
Name: bidirectional_first_seen_ms, dtype: int64
=== dollar_char test bidirectional_first_seen_ms ===
0       17
1      218
2      298
3    21778
4    26386
Name: bidirectional_first_seen_ms, dtype: int64


### Count attack type occurrences for each campaign
Inspect experiment details files to print the number of attack steps for each attacker action.

In [61]:
from collections import Counter

for name, (train_data, test_data) in zip(campaign_names, step_details_list):
    print(f"{name} train commands:", Counter(item["command"] for item in train_data))
    print(f"{name} test commands:", Counter(item["command"] for item in test_data))

dollar_char train commands: Counter({'pause': 284, 'dollar_char': 166, 'nmap_mqtt': 21, 'nmap_banner': 14, 'netstat': 9, 'nmap_10_T5': 9, 'nmap_192_T4': 8, 'nmap_sub': 7, 'mqtt_cat': 7, 'scp_inst': 7, 'brute_force_malformed': 1})
dollar_char test commands: Counter({'pause': 164, 'dollar_char': 63, 'nmap_mqtt': 29, 'nmap_banner': 20, 'mqtt_cat': 9, 'scp_inst': 9, 'netstat': 5, 'nmap_192_T4': 5, 'nmap_10_T5': 5, 'brute_force_malformed': 1})


### Uniform time zone for start timestamp and experiment timestamps
Load start_time from .npz file. This is the starting timestamp of the attack campaign.

In [62]:
start_time_list = []
for train_start, test_start in start_list:
    train_st = np.load(train_start, allow_pickle=True)["datetime"]
    test_st = np.load(test_start, allow_pickle=True)["datetime"]
    start_time_list.append((train_st, test_st))

In [63]:
for name, (train_st, test_st) in zip(campaign_names, start_time_list):
    print(f"{name} raw start_time -> train: {train_st}, test: {test_st}")

dollar_char raw start_time -> train: 2025-08-19 21:06:57.247039, test: 2025-08-20 13:35:53.809608


In [64]:
from preprocessing.time import shift_capture_time

start_time_list = [
    (shift_capture_time(train_st.item()), shift_capture_time(test_st.item()))
    for train_st, test_st in start_time_list
]

In [65]:
for name, (train_st, test_st) in zip(campaign_names, start_time_list):
    print(f"{name} shifted start_time -> train: {train_st}, test: {test_st}")

dollar_char shifted start_time -> train: 2025-08-19 19:06:57.247039, test: 2025-08-20 11:35:53.809608


### Convert relative timestamps to UTC

We convert the 'bidirectional_first_seen_ms' relative time to absolute UTC timestamps using as reference the start_time we loaded from the .npz file.

In [66]:
from preprocessing.time import absolute_flow_time

convert_to_absolute_time = absolute_flow_time
for (train_df, test_df), (train_st, test_st) in zip(df_list, start_time_list):
    train_df["timestamp"] = train_df["bidirectional_first_seen_ms"].apply(
        convert_to_absolute_time, start_time=train_st
    )
    test_df["timestamp"] = test_df["bidirectional_first_seen_ms"].apply(
        convert_to_absolute_time, start_time=test_st
    )

In [67]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"=== {name} train timestamps ===")
    print(train_df[["timestamp", "bidirectional_first_seen_ms"]].head())
    print(f"=== {name} test timestamps ===")
    print(test_df[["timestamp", "bidirectional_first_seen_ms"]].head())

=== dollar_char train timestamps ===
                         timestamp  bidirectional_first_seen_ms
0 2025-08-19 19:06:57.257039+00:00                           10
1 2025-08-19 19:06:57.595039+00:00                          348
2 2025-08-19 19:06:57.659039+00:00                          412
3 2025-08-19 19:06:57.707039+00:00                          460
4 2025-08-19 19:06:57.723039+00:00                          476
=== dollar_char test timestamps ===
                         timestamp  bidirectional_first_seen_ms
0 2025-08-20 11:35:53.826608+00:00                           17
1 2025-08-20 11:35:54.027608+00:00                          218
2 2025-08-20 11:35:54.107608+00:00                          298
3 2025-08-20 11:36:15.587608+00:00                        21778
4 2025-08-20 11:36:20.195608+00:00                        26386


### Assign labels
- We assign labels based on attack step time intervals: if an observation begins between the start_time and the end_time of a step, it is marked with the corresponding attack type name.
- Not labeled observations are marked as "discard": they correspond to sleep times between attack steps. 

In [68]:
step_list = []
for name, (train_details, test_details) in zip(campaign_names, step_details_list):
    train_step = pd.DataFrame(train_details)
    test_step = pd.DataFrame(test_details)
    step_list.append((train_step, test_step))
    print(f"=== {name} train step ({train_step.shape}) ===")
    print(train_step.head())
    print(f"=== {name} test step ({test_step.shape}) ===")
    print(test_step.head())

=== dollar_char train step ((533, 8)) ===
   phase_number  step_number                         attack_name  \
0             1            1  dollar_char_dos_scp_inst_cam_train   
1            -1            2  dollar_char_dos_scp_inst_cam_train   
2             1            3  dollar_char_dos_scp_inst_cam_train   
3            -1            4  dollar_char_dos_scp_inst_cam_train   
4             1            5  dollar_char_dos_scp_inst_cam_train   

       phase_name                       start_time  \
0  RECONNAISSANCE 2025-08-19 19:11:16.388096+00:00   
1           PAUSE 2025-08-19 19:11:16.576560+00:00   
2  RECONNAISSANCE 2025-08-19 19:12:23.815666+00:00   
3           PAUSE 2025-08-19 19:12:23.888061+00:00   
4  RECONNAISSANCE 2025-08-19 19:13:56.712583+00:00   

                          end_time      command  iteration  
0 2025-08-19 19:11:16.576507+00:00      netstat          0  
1 2025-08-19 19:12:23.815582+00:00        pause          0  
2 2025-08-19 19:12:23.888013+00:00  nmap_

In [69]:
for name, (train_step, test_step) in zip(campaign_names, step_list):
    print(f"=== {name} train step[0:10] ===")
    print(train_step.iloc[0:10])
    print(f"=== {name} test step[0:10] ===")
    print(test_step.iloc[0:10])

=== dollar_char train step[0:10] ===
   phase_number  step_number                         attack_name  \
0             1            1  dollar_char_dos_scp_inst_cam_train   
1            -1            2  dollar_char_dos_scp_inst_cam_train   
2             1            3  dollar_char_dos_scp_inst_cam_train   
3            -1            4  dollar_char_dos_scp_inst_cam_train   
4             1            5  dollar_char_dos_scp_inst_cam_train   
5            -1            6  dollar_char_dos_scp_inst_cam_train   
6             1            7  dollar_char_dos_scp_inst_cam_train   
7            -1            8  dollar_char_dos_scp_inst_cam_train   
8             1            9  dollar_char_dos_scp_inst_cam_train   
9            -1           10  dollar_char_dos_scp_inst_cam_train   

       phase_name                       start_time  \
0  RECONNAISSANCE 2025-08-19 19:11:16.388096+00:00   
1           PAUSE 2025-08-19 19:11:16.576560+00:00   
2  RECONNAISSANCE 2025-08-19 19:12:23.815666+00:00  

In [70]:
# Ensure tz-aware UTC
for (train_step, test_step), (train_df, test_df) in zip(step_list, df_list):
    for step in (train_step, test_step):
        step["start_time"] = pd.to_datetime(step["start_time"], utc=True)
        step["end_time"] = pd.to_datetime(step["end_time"], utc=True)
    for df in (train_df, test_df):
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

In [71]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"{name} train timestamp:", train_df["timestamp"].head())
    print(f"{name} test timestamp:", test_df["timestamp"].head())

dollar_char train timestamp: 0   2025-08-19 19:06:57.257039+00:00
1   2025-08-19 19:06:57.595039+00:00
2   2025-08-19 19:06:57.659039+00:00
3   2025-08-19 19:06:57.707039+00:00
4   2025-08-19 19:06:57.723039+00:00
Name: timestamp, dtype: datetime64[us, UTC]
dollar_char test timestamp: 0   2025-08-20 11:35:53.826608+00:00
1   2025-08-20 11:35:54.027608+00:00
2   2025-08-20 11:35:54.107608+00:00
3   2025-08-20 11:36:15.587608+00:00
4   2025-08-20 11:36:20.195608+00:00
Name: timestamp, dtype: datetime64[us, UTC]


In [72]:
for name, (train_step, test_step) in zip(campaign_names, step_list):
    print(f"{name} train end_time:", train_step["end_time"].head())
    print(f"{name} test end_time:", test_step["end_time"].head())

dollar_char train end_time: 0   2025-08-19 19:11:16.576507+00:00
1   2025-08-19 19:12:23.815582+00:00
2   2025-08-19 19:12:23.888013+00:00
3   2025-08-19 19:13:56.712519+00:00
4   2025-08-19 19:14:52.953673+00:00
Name: end_time, dtype: datetime64[us, UTC]
dollar_char test end_time: 0   2025-08-20 11:38:18.881681+00:00
1   2025-08-20 11:40:47.329054+00:00
2   2025-08-20 11:40:47.408082+00:00
3   2025-08-20 11:43:23.570797+00:00
4   2025-08-20 11:44:19.527836+00:00
Name: end_time, dtype: datetime64[us, UTC]


In [73]:
for name, (train_step, test_step) in zip(campaign_names, step_list):
    print(f"{name} train start_time:", train_step["start_time"].head())
    print(f"{name} test start_time:", test_step["start_time"].head())

dollar_char train start_time: 0   2025-08-19 19:11:16.388096+00:00
1   2025-08-19 19:11:16.576560+00:00
2   2025-08-19 19:12:23.815666+00:00
3   2025-08-19 19:12:23.888061+00:00
4   2025-08-19 19:13:56.712583+00:00
Name: start_time, dtype: datetime64[us, UTC]
dollar_char test start_time: 0   2025-08-20 11:38:18.722238+00:00
1   2025-08-20 11:38:18.881726+00:00
2   2025-08-20 11:40:47.329176+00:00
3   2025-08-20 11:40:47.408113+00:00
4   2025-08-20 11:43:23.570862+00:00
Name: start_time, dtype: datetime64[us, UTC]


In [74]:
iv_list = [
    (
        pd.IntervalIndex.from_arrays(train_step["start_time"], train_step["end_time"], closed="both"),
        pd.IntervalIndex.from_arrays(test_step["start_time"], test_step["end_time"], closed="both"),
    )
    for train_step, test_step in step_list
]

In [75]:
for name, (train_iv, test_iv) in zip(campaign_names, iv_list):
    print(f"{name} train iv ({len(train_iv)}):", train_iv[:5])
    print(f"{name} test iv ({len(test_iv)}):", test_iv[:5])

dollar_char train iv (533): IntervalIndex([[2025-08-19 19:11:16.388096+00:00, 2025-08-19 19:11:16.576507+00:00],
               [2025-08-19 19:11:16.576560+00:00, 2025-08-19 19:12:23.815582+00:00],
               [2025-08-19 19:12:23.815666+00:00, 2025-08-19 19:12:23.888013+00:00],
               [2025-08-19 19:12:23.888061+00:00, 2025-08-19 19:13:56.712519+00:00],
               [2025-08-19 19:13:56.712583+00:00, 2025-08-19 19:14:52.953673+00:00]],
              dtype='interval[datetime64[us, UTC], both]')
dollar_char test iv (310): IntervalIndex([[2025-08-20 11:38:18.722238+00:00, 2025-08-20 11:38:18.881681+00:00],
               [2025-08-20 11:38:18.881726+00:00, 2025-08-20 11:40:47.329054+00:00],
               [2025-08-20 11:40:47.329176+00:00, 2025-08-20 11:40:47.408082+00:00],
               [2025-08-20 11:40:47.408113+00:00, 2025-08-20 11:43:23.570797+00:00],
               [2025-08-20 11:43:23.570862+00:00, 2025-08-20 11:44:19.527836+00:00]],
              dtype='interval[date

In [76]:
idx_list = [
    (
        train_iv.get_indexer(train_df["timestamp"]),
        test_iv.get_indexer(test_df["timestamp"]),
    )
    for (train_iv, test_iv), (train_df, test_df) in zip(iv_list, df_list)
]

In [77]:
for name, (train_idx, test_idx) in zip(campaign_names, idx_list):
    print(f"{name} train idx (first 20):", train_idx[:20])
    print(f"{name} test idx (first 20):", test_idx[:20])

dollar_char train idx (first 20): [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
dollar_char test idx (first 20): [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]


In [80]:
for name, (train_df, test_df), (train_step, test_step), (train_idx, test_idx) in zip(
    campaign_names, df_list, step_list, idx_list
):
    cid = campaign_ids[name]
    for df, step, idx in ((train_df, train_step, train_idx), (test_df, test_step, test_idx)):
        df["campaign_id"] = cid
        df["phase_idx"] = idx
        df["phase_name"] = np.where(idx >= 0, step["phase_name"].to_numpy()[idx], None)
        df["label"] = np.where(idx >= 0, step["command"].to_numpy()[idx], "discard")
        df["phase_number"] = np.where(idx >= 0, step["phase_number"].to_numpy()[idx], None)
        df["step_number"] = np.where(idx >= 0, step["step_number"].to_numpy()[idx], None)

In [81]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"=== {name} train label counts ===")
    print(train_df["label"].value_counts())
    print(f"=== {name} test label counts ===")
    print(test_df["label"].value_counts())

=== dollar_char train label counts ===
label
nmap_10_T5               189139
pause                      3667
dollar_char                1995
nmap_mqtt                  1797
nmap_banner                1070
brute_force_malformed       659
nmap_sub                    368
discard                     217
mqtt_cat                      9
scp_inst                      7
Name: count, dtype: int64
=== dollar_char test label counts ===
label
nmap_10_T5               105089
pause                      4797
nmap_mqtt                  2498
nmap_banner                1555
brute_force_malformed       596
dollar_char                 516
discard                     146
mqtt_cat                     11
scp_inst                     10
nmap_192_T4                   1
Name: count, dtype: int64


In [82]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"=== {name} train phase_name counts ===")
    print(train_df["phase_name"].value_counts())
    print(f"=== {name} test phase_name counts ===")
    print(test_df["phase_name"].value_counts())

=== dollar_char train phase_name counts ===
phase_name
RECONNAISSANCE    189139
PAUSE               3667
DISCOVERY           3244
EXPLOIT             1995
BRUTE_FORCE          659
INSTALLATION           7
Name: count, dtype: int64
=== dollar_char test phase_name counts ===
phase_name
RECONNAISSANCE    105090
PAUSE               4797
DISCOVERY           4064
BRUTE_FORCE          596
EXPLOIT              516
INSTALLATION          10
Name: count, dtype: int64


In [83]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    for split, df in (("train", train_df), ("test", test_df)):
        print(f"=== {name} {split}: step number - {len(df['step_number'].value_counts().keys())} ===")
        for step_name, value in df["step_number"].value_counts().items():
            print(f"{step_name}:\t{value}")

=== dollar_char train: step number - 513 ===
17:	21050
5:	21017
35:	21014
11:	21010
23:	21010
219:	21010
231:	21010
29:	21009
225:	21009
37:	659
421:	160
450:	95
277:	88
69:	87
121:	87
333:	87
478:	87
77:	86
81:	86
149:	86
249:	86
506:	86
177:	85
71:	84
73:	84
79:	84
75:	84
305:	84
389:	84
205:	83
417:	83
361:	81
207:	80
179:	79
452:	79
480:	78
123:	76
251:	76
279:	76
335:	76
363:	76
151:	75
391:	75
419:	75
508:	75
307:	74
6:	65
63:	55
276:	55
159:	54
39:	53
55:	53
43:	52
51:	52
59:	52
62:	52
383:	52
47:	51
329:	51
106:	49
215:	46
482:	36
427:	30
426:	27
481:	27
16:	26
346:	23
68:	22
74:	21
326:	21
404:	21
52:	20
280:	20
14:	19
64:	19
124:	19
235:	19
240:	19
342:	19
32:	18
66:	18
98:	18
100:	18
298:	18
318:	18
504:	18
2:	17
4:	17
72:	17
132:	17
176:	17
234:	17
247:	17
264:	17
284:	17
352:	17
378:	17
380:	17
429:	17
461:	17
477:	17
507:	17
509:	17
515:	17
523:	17
34:	16
50:	16
82:	16
94:	16
110:	16
120:	16
140:	16
152:	16
169:	16
172:	16
182:	16
200:	16
212:	16
220:	16
228:	16
250:	16
3

## Summary tables
Print a summary of the dataframe we built to evaluate correctness of the previous operations.

In [84]:
from preprocessing.datasets import summarize_labeled_flows

step_summary_list = []
label_summary_list = []
for i, (train_df, test_df) in enumerate(df_list):
    train_df, train_step_summary, train_label_summary = summarize_labeled_flows(train_df)
    test_df, test_step_summary, test_label_summary = summarize_labeled_flows(test_df)
    df_list[i] = (train_df, test_df)
    step_summary_list.append((train_step_summary, test_step_summary))
    label_summary_list.append((train_label_summary, test_label_summary))

In [85]:
for name, (train_label_summary, test_label_summary) in zip(campaign_names, label_summary_list):
    print(f"=== {name} train label_summary ===")
    display(train_label_summary)
    print(f"=== {name} test label_summary ===")
    display(test_label_summary)

=== dollar_char train label_summary ===


,label,avg_num_flws,median_num_flws,avg_duration,median_duration,total_steps,total_flws,avg_fps
0,brute_force_malformed,659.000000,659.0,0 days 00:14:49.435000,0 days 00:14:49.435000,1,659,0.740920
1,dollar_char,12.018072,10.0,0 days 00:00:52.558548,0 days 00:00:47.397000,166,1995,0.233225
2,mqtt_cat,1.500000,1.0,0 days 00:00:01.198666,0 days 00:00:00,6,9,1.467646
3,nmap_10_T5,21015.444444,21010.0,0 days 00:00:52.132777,0 days 00:00:51.614000,9,189139,403.857130
4,nmap_banner,76.428571,76.0,0 days 00:01:00.220071,0 days 00:01:00.698500,14,1070,1.270563
5,nmap_mqtt,85.571429,86.0,0 days 00:03:57.972571,0 days 00:03:57.646000,21,1797,0.359606
6,nmap_sub,52.571429,52.0,0 days 00:00:52.265000,0 days 00:00:52.042000,7,368,1.006442
7,pause,13.003546,12.0,0 days 00:01:02.146588,0 days 00:01:02.526500,282,3667,0.212957
8,scp_inst,1.000000,1.0,0 days 00:00:00,0 days 00:00:00,7,7,NaN


=== dollar_char test label_summary ===


,label,avg_num_flws,median_num_flws,avg_duration,median_duration,total_steps,total_flws,avg_fps
0,brute_force_malformed,596.000000,596.0,0 days 00:14:52.988000,0 days 00:14:52.988000,1,596,0.667422
1,dollar_char,8.190476,7.0,0 days 00:00:36.031682,0 days 00:00:26.966000,63,516,0.285905
2,mqtt_cat,1.375000,1.0,0 days 00:00:00.824125,0 days 00:00:00,8,11,1.119125
3,nmap_10_T5,21017.800000,21012.0,0 days 00:00:52.774000,0 days 00:00:53.478000,5,105089,399.136671
4,nmap_192_T4,1.000000,1.0,0 days 00:00:00,0 days 00:00:00,1,1,NaN
5,nmap_banner,77.750000,76.5,0 days 00:01:01.399050,0 days 00:01:01.758000,20,1555,1.268086
6,nmap_mqtt,86.137931,85.0,0 days 00:03:58.706000,0 days 00:03:58.915000,29,2498,0.360859
7,pause,29.250000,27.0,0 days 00:02:20.908237,0 days 00:02:21.189500,164,4797,0.206996
8,scp_inst,1.111111,1.0,0 days 00:00:00.004555,0 days 00:00:00,9,10,48.780488


In [86]:
for name, (train_step_summary, test_step_summary) in zip(campaign_names, step_summary_list):
    print(f"=== {name} train step_summary ===")
    display(train_step_summary)
    print(f"=== {name} test step_summary ===")
    display(test_step_summary)

=== dollar_char train step_summary ===


,step_number,label,phase_name,phase_number,num_flws,start_time,end_time,duration,fps
0,2,pause,PAUSE,-1,17,2025-08-19 19:11:17.475039+00:00,2025-08-19 19:12:18.459039+00:00,0 days 00:01:00.984000,0.278762
1,4,pause,PAUSE,-1,17,2025-08-19 19:12:24.506039+00:00,2025-08-19 19:13:54.169039+00:00,0 days 00:01:29.663000,0.189599
2,5,nmap_10_T5,RECONNAISSANCE,1,21017,2025-08-19 19:14:01.213039+00:00,2025-08-19 19:14:52.827039+00:00,0 days 00:00:51.614000,407.195722
3,6,pause,PAUSE,-1,65,2025-08-19 19:14:54.563039+00:00,2025-08-19 19:16:07.065039+00:00,0 days 00:01:12.502000,0.896527
4,8,pause,PAUSE,-1,10,2025-08-19 19:16:13.117039+00:00,2025-08-19 19:16:57.445039+00:00,0 days 00:00:44.328000,0.225591
...,...,...,...,...,...,...,...,...,...
508,527,pause,PAUSE,-1,7,2025-08-20 05:03:02.231039+00:00,2025-08-20 05:03:32.219039+00:00,0 days 00:00:29.988000,0.233427
509,528,dollar_char,EXPLOIT,5,8,2025-08-20 05:03:32.470039+00:00,2025-08-20 05:04:16.718039+00:00,0 days 00:00:44.248000,0.180799
510,529,pause,PAUSE,-1,11,2025-08-20 05:04:24.760039+00:00,2025-08-20 05:05:21.969039+00:00,0 days 00:00:57.209000,0.192277
511,530,dollar_char,EXPLOIT,5,9,2025-08-20 05:05:22.210039+00:00,2025-08-20 05:06:08.451039+00:00,0 days 00:00:46.241000,0.194632


=== dollar_char test step_summary ===


,step_number,label,phase_name,phase_number,num_flws,start_time,end_time,duration,fps
0,2,pause,PAUSE,-1,83,2025-08-20 11:38:23.472608+00:00,2025-08-20 11:40:41.336608+00:00,0 days 00:02:17.864000,0.602043
1,3,nmap_192_T4,RECONNAISSANCE,1,1,2025-08-20 11:40:47.375608+00:00,2025-08-20 11:40:47.375608+00:00,0 days 00:00:00,NaN
2,4,pause,PAUSE,-1,30,2025-08-20 11:40:55.427608+00:00,2025-08-20 11:43:21.458608+00:00,0 days 00:02:26.031000,0.205436
3,5,nmap_10_T5,RECONNAISSANCE,1,21036,2025-08-20 11:43:24.644608+00:00,2025-08-20 11:44:19.399608+00:00,0 days 00:00:54.755000,384.184093
4,6,pause,PAUSE,-1,62,2025-08-20 11:44:25.904608+00:00,2025-08-20 11:47:05.997608+00:00,0 days 00:02:40.093000,0.387275
...,...,...,...,...,...,...,...,...,...
295,306,pause,PAUSE,-1,29,2025-08-20 21:28:05.246608+00:00,2025-08-20 21:30:47.224608+00:00,0 days 00:02:41.978000,0.179037
296,307,dollar_char,EXPLOIT,5,7,2025-08-20 21:30:47.474608+00:00,2025-08-20 21:31:22.648608+00:00,0 days 00:00:35.174000,0.199011
297,308,pause,PAUSE,-1,28,2025-08-20 21:31:28.696608+00:00,2025-08-20 21:34:05.227608+00:00,0 days 00:02:36.531000,0.178878
298,309,dollar_char,EXPLOIT,5,6,2025-08-20 21:34:05.467608+00:00,2025-08-20 21:34:34.991608+00:00,0 days 00:00:29.524000,0.203224


## Remove Pauses 

In [87]:
df_list = [
    (
        train_df[(train_df["label"] != "pause") & (train_df["label"] != "discard")].reset_index(drop=True),
        test_df[(test_df["label"] != "pause") & (test_df["label"] != "discard")].reset_index(drop=True),
    )
    for train_df, test_df in df_list
]

In [88]:
for name, (train_df, test_df) in zip(campaign_names, df_list):
    print(f"{name} train shape after removing pause/discard: {train_df.shape}")
    print(f"{name} test shape after removing pause/discard: {test_df.shape}")

dollar_char train shape after removing pause/discard: (195044, 93)
dollar_char test shape after removing pause/discard: (110276, 93)


## Save labeled df to CSV

In [89]:
for name, (train_root, test_root), (train_df, test_df) in zip(campaign_names, root_list, df_list):
    train_root.mkdir(parents=True, exist_ok=True)
    test_root.mkdir(parents=True, exist_ok=True)
    train_out = train_root / f"{train_root.name}_labeled.csv"
    test_out = test_root / f"{test_root.name}_labeled.csv"
    train_df.to_csv(train_out, index=False)
    test_df.to_csv(test_out, index=False)
    print(f"Saved {train_out} ({train_df.shape})")
    print(f"Saved {test_out} ({test_df.shape})")

Saved /home/users/davide.zhang/updated_attack_scenario_reconstruction_project/Attack-Scenario-Reconstruction/data/cAPTure/attack/dollar_char_train/dollar_char_train_labeled.csv ((195044, 93))
Saved /home/users/davide.zhang/updated_attack_scenario_reconstruction_project/Attack-Scenario-Reconstruction/data/cAPTure/attack/dollar_char_test/dollar_char_test_labeled.csv ((110276, 93))
